# 2. Open Queuing network simulation (8 pt)
The second part of the project is to make an open-queuing network simulator, where the jobs arrive from the outside following Poisson processes with the rate of lambda. The system is shown as the following figure:

![image](Figure.png)

Each job needs to go to the CPU, following an exponential distribution with a mean of 10 jobs per second, and then proceeds by either one of the disks. There are two disks, each of which has an exponentially-distributed processing time. The fast disk has a processing rate of 12 jobs per second and the slow disk has the rate of 9 jobs per second. The number of buffer spaces in each queue is infinite.


# Questions:
(4pt) Use your simulator to find the maximum sustainable throughput of such a system in terms of number of jobs. Support your answer with simulation results.

(4pt) Find out the average response times for following cases over three arrival rates of your choice: case (i) - a single queue in front of fast and slow disk; case (ii) - a separate queue for each disk, jointly applying the shortest queue load balancing strategy when sending the jobs from the CPU to disk.

![image](Figure_again.png)


In [1]:
import simpy
import numpy as np

class OpenNetwork:
    def __init__(self, env, cpu_rate, slow_rate, fast_rate):
        self.env = env
        self.cpu = simpy.Resource(env, capacity=1)
        self.slow_disk = simpy.Resource(env, capacity=1)
        self.fast_disk = simpy.Resource(env, capacity=1)
        self.cpu_rate = cpu_rate
        self.slow_rate = slow_rate
        self.fast_rate = fast_rate
        self.done = 0

    def job(self):
        with self.cpu.request() as req:
            yield req
            yield self.env.timeout(np.random.exponential(1/self.cpu_rate))
        # Pick disk at random
        if np.random.random() < 0.5:
            with self.slow_disk.request() as req:
                yield req
                yield self.env.timeout(np.random.exponential(1/self.slow_rate))
        else:
            with self.fast_disk.request() as req:
                yield req
                yield self.env.timeout(np.random.exponential(1/self.fast_rate))
        self.done += 1

def generate_jobs(env, system, lam):
    while True:
        yield env.timeout(np.random.exponential(1/lam))
        env.process(system.job())

def simulate(lam, sim_time=10000):
    env = simpy.Environment()
    system = OpenNetwork(env, cpu_rate=10, slow_rate=9, fast_rate=12)
    env.process(generate_jobs(env, system, lam))
    env.run(until=sim_time)
    return system.done / sim_time

# Test a range of λ
results = []
for lam in np.arange(5, 15, 1):
    avg_throughput = np.mean([simulate(lam) for _ in range(3)])
    results.append((lam, avg_throughput))
print(results)


[(np.int64(5), np.float64(5.016066666666666)), (np.int64(6), np.float64(5.989999999999999)), (np.int64(7), np.float64(7.0088)), (np.int64(8), np.float64(8.012833333333333)), (np.int64(9), np.float64(9.015299999999998)), (np.int64(10), np.float64(9.9603)), (np.int64(11), np.float64(10.002033333333335)), (np.int64(12), np.float64(9.964766666666666)), (np.int64(13), np.float64(10.0108)), (np.int64(14), np.float64(10.001100000000001))]


In [2]:
from typing import Generator, Literal
import numpy as np
import simpy

RNG = np.random.default_rng(seed=0)

class OpenNetSplitDisks:
    """
    Open network with:
      - Poisson arrivals rate lambda_
      - 1 CPU: Exp service rate mucpu
      - 2 disks (fast, slow): Exp service rates mufast, muslow
        Each job: CPU -> (fast or slow) with prob 0.5 each, then leaves.
    """
    def __init__(self, lambda_: float, mucpu: float, mufast: float, muslow: float) -> None:
        self.env = simpy.Environment()
        self.beta_arr = 1.0 / lambda_
        self.beta_cpu = 1.0 / mucpu
        self.beta_fast = 1.0 / mufast
        self.beta_slow = 1.0 / muslow

        self.cpu = simpy.Resource(self.env, capacity=1)
        self.fast = simpy.Resource(self.env, capacity=1)
        self.slow = simpy.Resource(self.env, capacity=1)

        # stats
        self.n_curr_jobs = 0
        self.log_jobs_in_sys: list[int] = []

        self.et_acc = 0.0
        self.n_served_jobs = 0

        self.rho_cpu_acc = 0.0
        self.rho_fast_acc = 0.0
        self.rho_slow_acc = 0.0
        self.rho_sys_acc = 0.0

    # one job through CPU then one of the disks
    def system_generator(self) -> Generator:
        job_start_time = self.env.now

        # snapshot utilizations
        rho_cpu = self.cpu.count / self.cpu.capacity
        rho_fast = self.fast.count / self.fast.capacity
        rho_slow = self.slow.count / self.slow.capacity
        self.rho_cpu_acc += rho_cpu
        self.rho_fast_acc += rho_fast
        self.rho_slow_acc += rho_slow
        self.rho_sys_acc += (rho_cpu or rho_fast or rho_slow)

        self.log_jobs_in_sys.append(self.n_curr_jobs)
        self.n_curr_jobs += 1

        # CPU
        with self.cpu.request() as req:
            yield req
            yield self.env.timeout(RNG.exponential(scale=self.beta_cpu))

        # routing to disks (50/50)
        if RNG.random() < 0.5:
            disk = self.fast
            beta = self.beta_fast
        else:
            disk = self.slow
            beta = self.beta_slow

        with disk.request() as req:
            yield req
            yield self.env.timeout(RNG.exponential(scale=beta))

        self.n_curr_jobs -= 1
        self.et_acc += self.env.now - job_start_time
        self.n_served_jobs += 1

    # Poisson arrivals
    def job_generator(self) -> Generator:
        while True:
            yield self.env.timeout(RNG.exponential(self.beta_arr))
            self.env.process(self.system_generator())

    def run(self, t: float) -> "OpenNetSplitDisks":
        self.env.process(self.job_generator())
        self.env.run(until=t)
        return self

    # metrics (same style as OpenNet)
    def system_throughput(self) -> float:
        return self.n_served_jobs / self.env.now

    def utilization(self, comp: Literal["cpu", "fast", "slow", "sys"]) -> float:
        acc = {
            "cpu": self.rho_cpu_acc,
            "fast": self.rho_fast_acc,
            "slow": self.rho_slow_acc,
            "sys": self.rho_sys_acc,
        }[comp]
        return acc / len(self.log_jobs_in_sys)


In [3]:
def simulate_for_lambda(lambda_, sim_time=1_000_000.0):
    mucpu = 10.0         # CPU rate
    mufast = 12.0        # fast disk rate
    muslow = 9.0         # slow disk rate

    net = OpenNetSplitDisks(lambda_, mucpu, mufast, muslow).run(sim_time)
    return {
        "lambda": lambda_,
        "throughput": net.system_throughput(),
        "rho_cpu": net.utilization("cpu"),
        "rho_fast": net.utilization("fast"),
        "rho_slow": net.utilization("slow"),
        "rho_sys": net.utilization("sys"),
    }

lambdas = [7.0, 8.0, 8.5, 9.0, 9.5, 10.0]
results = [simulate_for_lambda(lmbd) for lmbd in lambdas]

for r in results:
    print(
        f"lambda={r['lambda']:4.1f}  "
        f"X={r['throughput']:.3f}  "
        f"rho_cpu={r['rho_cpu']:.3f}  "
        f"rho_fast={r['rho_fast']:.3f}  "
        f"rho_slow={r['rho_slow']:.3f}  "
        f"rho_sys={r['rho_sys']:.3f}"
    )


lambda= 7.0  X=6.995  rho_cpu=0.699  rho_fast=0.291  rho_slow=0.388  rho_sys=0.870
lambda= 8.0  X=8.002  rho_cpu=0.800  rho_fast=0.334  rho_slow=0.445  rho_sys=0.926
lambda= 8.5  X=8.500  rho_cpu=0.850  rho_fast=0.354  rho_slow=0.473  rho_sys=0.949
lambda= 9.0  X=9.000  rho_cpu=0.900  rho_fast=0.375  rho_slow=0.500  rho_sys=0.969
lambda= 9.5  X=9.501  rho_cpu=0.951  rho_fast=0.396  rho_slow=0.528  rho_sys=0.986
lambda=10.0  X=9.997  rho_cpu=0.999  rho_fast=0.417  rho_slow=0.555  rho_sys=1.000


In [4]:
from typing import Generator, Literal
import numpy as np
import simpy

RNG = np.random.default_rng(seed=0)

class OpenNetSingleQueue:
    def __init__(self, lambda_, mucpu, mufast, muslow):
        self.env = simpy.Environment()
        self.beta_arr = 1.0 / lambda_
        self.beta_cpu = 1.0 / mucpu
        self.beta_fast = 1.0 / mufast
        self.beta_slow = 1.0 / muslow

        self.cpu = simpy.Resource(self.env, capacity=1)
        # single waiting line, but two disk servers behind it
        self.disk_queue = simpy.Store(self.env)
        self.fast = simpy.Resource(self.env, capacity=1)
        self.slow = simpy.Resource(self.env, capacity=1)

        self.et_acc = 0.0
        self.n_served_jobs = 0

        # start disk server processes
        self.env.process(self.disk_server(self.fast, self.beta_fast))
        self.env.process(self.disk_server(self.slow, self.beta_slow))

    def disk_server(self, disk_res: simpy.Resource, beta_disk: float) -> Generator:
        while True:
            job_start_time = yield self.disk_queue.get()   # get (start_time) from shared queue
            with disk_res.request() as req:
                yield req
                yield self.env.timeout(RNG.exponential(scale=beta_disk))
            # record response time
            self.et_acc += self.env.now - job_start_time
            self.n_served_jobs += 1

    def system_generator(self) -> Generator:
        job_start_time = self.env.now
        # CPU
        with self.cpu.request() as req:
            yield req
            yield self.env.timeout(RNG.exponential(scale=self.beta_cpu))
        # enqueue to common disk queue
        yield self.disk_queue.put(job_start_time)

    def job_generator(self) -> Generator:
        while True:
            yield self.env.timeout(RNG.exponential(self.beta_arr))
            self.env.process(self.system_generator())

    def run(self, T: float) -> "OpenNetSingleQueue":
        self.env.process(self.job_generator())
        self.env.run(until=T)
        return self

    def mean_response_time(self) -> float:
        return self.et_acc / self.n_served_jobs


In [5]:
def simulate_singlequeue_for_lambdas(lambdas, T=1_000_000.0):
    mucpu, mufast, muslow = 10.0, 12.0, 9.0
    for lam in lambdas:
        net = OpenNetSingleQueue(lam, mucpu, mufast, muslow).run(T)
        print(f"[single queue] lambda={lam:.1f},  E[T]={net.mean_response_time():.4f}")

lambdas = [5.0, 7.0, 8.5]
simulate_singlequeue_for_lambdas(lambdas)


[single queue] lambda=5.0,  E[T]=0.3023
[single queue] lambda=7.0,  E[T]=0.4416
[single queue] lambda=8.5,  E[T]=0.7806


In [ ]:
class OpenNetShortestQueue:
    """
    Two separate disk queues; jobs from CPU join the shorter queue
    (ties broken randomly).
    """
    def __init__(self, lambda_, mucpu, mufast, muslow):
        self.env = simpy.Environment()
        self.beta_arr = 1.0 / lambda_
        self.beta_cpu = 1.0 / mucpu
        self.beta_fast = 1.0 / mufast
        self.beta_slow = 1.0 / muslow

        self.cpu = simpy.Resource(self.env, capacity=1)
        self.fast = simpy.Resource(self.env, capacity=1)
        self.slow = simpy.Resource(self.env, capacity=1)

        self.et_acc = 0.0
        self.n_served_jobs = 0

    def job_flow(self) -> Generator:
        job_start_time = self.env.now

        # CPU
        with self.cpu.request() as req:
            yield req
            yield self.env.timeout(RNG.exponential(scale=self.beta_cpu))

        # choose disk with shortest queue (including job in service)
        q_fast = len(self.fast.queue) + self.fast.count
        q_slow = len(self.slow.queue) + self.slow.count
        if q_fast < q_slow:
            disk, beta = self.fast, self.beta_fast
        elif q_slow < q_fast:
            disk, beta = self.slow, self.beta_slow
        else:  # tie
            if RNG.random() < 0.5:
                disk, beta = self.fast, self.beta_fast
            else:
                disk, beta = self.slow, self.beta_slow

        with disk.request() as req:
            yield req
            yield self.env.timeout(RNG.exponential(scale=beta))

        self.et_acc += self.env.now - job_start_time
        self.n_served_jobs += 1

    def job_generator(self) -> Generator:
        while True:
            yield self.env.timeout(RNG.exponential(self.beta_arr))
            self.env.process(self.job_flow())

    def run(self, T: float) -> "OpenNetShortestQueue":
        self.env.process(self.job_generator())
        self.env.run(until=T)
        return self

    def mean_response_time(self) -> float:
        return self.et_acc / self.n_served_jobs


In [ ]:
def simulate_shortestqueue_for_lambdas(lambdas, T=1_000_000.0):
    mucpu, mufast, muslow = 10.0, 12.0, 9.0
    for lam in lambdas:
        net = OpenNetShortestQueue(lam, mucpu, mufast, muslow).run(T)
        print(f"[shortest queue] lambda={lam:.1f},  E[T]={net.mean_response_time():.4f}")

simulate_shortestqueue_for_lambdas(lambdas)
